The dataset contains time-series measurements from 100 simulated upstream bioprocess experiments for monoclonal antibody (mAb) production.  
Each row corresponds to one day of one experiment.

Column names follow the standard bioprocess convention with three prefixes:

- **`Z:`**  *Experimental design / setpoints*. Time-invariant parameters that define the experimental conditions. Only populated on day 0 of each experiment (NaN otherwise).
- **`W:`**  *Process inputs / actuators*. Controlled, time-dependent variables (the recipe being executed each day).
- **`X:`**  *State variables*. Measured bioprocess quantities that evolve over time.

### Identifiers

| Column | Meaning |
|---|---|
| `RowID` | Unique row identifier across the full dataset. |
| `Exp` | Experiment identifier (`Exp 1` … `Exp 100`). |
| `Time[day]` | Day of the experiment (0 = inoculation). |

### `Z:` — Experimental design (constants per experiment, set on day 0)

| Column | Meaning |
|---|---|
| `Z:FeedStart` | Day on which nutrient feeding begins. |
| `Z:FeedEnd` | Day on which nutrient feeding ends. |
| `Z:FeedRateGlc` | Glucose feed rate applied during the feeding window. |
| `Z:FeedRateGln` | Glutamine feed rate applied during the feeding window. |
| `Z:phStart` | Initial pH setpoint (before the pH shift). |
| `Z:phEnd` | pH setpoint after the shift. |
| `Z:phShift` | Day on which pH is shifted from `phStart` to `phEnd`. |
| `Z:tempStart` | Initial temperature setpoint (°C). |
| `Z:tempEnd` | Temperature setpoint after the shift (°C). |
| `Z:tempShift` | Day on which temperature is shifted. |
| `Z:Stir` | Stirring / agitation rate (rpm). |
| `Z:DO` | Dissolved oxygen setpoint (%). |
| `Z:ExpDuration` | Total planned duration of the experiment (days). |

### `W:` — Process inputs (daily controlled variables)

| Column | Meaning |
|---|---|
| `W:temp` | Actual temperature applied on that day (reflects the `tempStart` → `tempEnd` shift). |
| `W:pH` | Actual pH applied on that day (reflects the `phStart` → `phEnd` shift). |
| `W:FeedGlc` | Glucose fed on that day (zero outside the feeding window). |
| `W:FeedGln` | Glutamine fed on that day (zero outside the feeding window). |

### `X:` — State variables (daily measurements of the culture)

| Column | Meaning |
|---|---|
| `X:VCD` | Viable Cell Density (10⁶ cells/mL) — concentration of living cells. |
| `X:Glc` | Glucose concentration in the bioreactor (mM or g/L). |
| `X:Gln` | Glutamine concentration. |
| `X:Amm` | Ammonia concentration (toxic metabolic byproduct of Gln consumption). |
| `X:Lac` | Lactate concentration (metabolic byproduct of Glc consumption). |
| `X:Lysed` | Fraction of lysed (dead, ruptured) cells. |

### Target

The prediction target is **final product titer** (mAb concentration at the end of the experiment, one scalar per experiment)

In [ ]:
import pandas as pd

In [18]:
pd.set_option('display.max_columns', None)

In [19]:
train_data_path = "../data/datahow_interview_train_data.csv"
train_targets_data_path = "../data/datahow_interview_train_targets.csv"

In [20]:
train_df = pd.read_csv(train_data_path)
targets_df = pd.read_csv(train_targets_data_path)

In [21]:
# Drop RowID column
train_df = train_df.drop(columns="RowID")
targets_df = targets_df.drop(columns="RowID")

### Data Overview

In [22]:
train_df.head()

,Exp,Time[day],Z:FeedStart,Z:FeedEnd,Z:FeedRateGlc,Z:FeedRateGln,Z:phStart,Z:phEnd,Z:phShift,Z:tempStart,Z:tempEnd,Z:tempShift,Z:Stir,Z:DO,Z:ExpDuration,W:temp,W:pH,W:FeedGlc,W:FeedGln,X:VCD,X:Glc,X:Gln,X:Amm,X:Lac,X:Lysed
0,Exp 1,0,2.0,10.0,5.656566,6.818182,6.808081,6.479798,9.0,37.282828,35.070707,13.0,224.242424,43.383838,10.0,37.282828,6.808081,0.000000,0.000000,1.682644,2.853045,6.005418,0.100000,0.100000,0.000000
1,Exp 1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.282828,6.808081,0.000000,0.000000,3.617721,2.290199,3.598475,0.324724,1.348831,0.002831
2,Exp 1,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.282828,6.808081,5.656566,6.818182,6.431879,0.752652,0.481756,0.687781,3.065268,0.000884
3,Exp 1,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.282828,6.808081,5.656566,6.818182,10.279057,4.276583,2.516062,0.972871,4.196027,0.000000
4,Exp 1,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.282828,6.808081,5.656566,6.818182,14.888965,7.350100,3.471558,1.322927,4.845310,0.000218


In [23]:
print(f"Training data has {len(train_df)} rows and {train_df.shape[1]} columns")

Training data has 990 rows and 25 columns


In [24]:
num_experiments = train_df["Exp"].nunique()
print(f"Number of experiments: {num_experiments}")

Number of experiments: 100


## Setpoints (`Z:` columns)

The `Z:` columns describe the **experimental design**, the recipe chosen for each experiment (feed window, pH/temperature shifts, agitation, DO, planned duration).

They are constants of the experiment rather than measurements, so they are recorded **only once, on day 0**, and are `NaN` on every subsequent day. The cell below confirms this empirically.

In [29]:
setpoints_cols = [col for col in train_df.columns if col.startswith("Z:")]
setpoints_cols

['Z:FeedStart',
 'Z:FeedEnd',
 'Z:FeedRateGlc',
 'Z:FeedRateGln',
 'Z:phStart',
 'Z:phEnd',
 'Z:phShift',
 'Z:tempStart',
 'Z:tempEnd',
 'Z:tempShift',
 'Z:Stir',
 'Z:DO',
 'Z:ExpDuration']

#### Validate Setpoints

In [30]:
rows_with_setpoints = train_df.dropna(subset=setpoints_cols, how="all")
days_with_setpoints = rows_with_setpoints["Time[day]"].unique()

print(f"Days on which setpoints are recorded: {days_with_setpoints}")
print(f"Experiments with setpoints recorded: {len(rows_with_setpoints)} / {train_df['Exp'].nunique()}")

assert list(days_with_setpoints) == [0], "Setpoints found on days other than day 0"
assert len(rows_with_setpoints) == num_experiments, "Some experiments are missing setpoints"

Days on which setpoints are recorded: [0]
Experiments with setpoints recorded: 100 / 100


#### Validate PH Shift

In [31]:
# For each experiment, verify W:pH behavior matches the scheduled Z:phShift:
#   - 0 < phShift <= ExpDuration: W:pH must change on that day vs. the previous day.
#   - phShift > ExpDuration:      W:pH must stay constant throughout the experiment.
# (phShift == 0 is excluded: no preceding day to compare to.)

bad_exps = []
for exp, exp_df in train_df.groupby("Exp"):
    setpoints = exp_df.loc[exp_df["Time[day]"] == 0].iloc[0]
    ph_shift = int(setpoints["Z:phShift"])
    exp_duration = setpoints["Z:ExpDuration"]
    pH = exp_df.set_index("Time[day]")["W:pH"]

    if 0 < ph_shift <= exp_duration:
        if pH[ph_shift] == pH[ph_shift - 1]:
            bad_exps.append(exp)
    elif ph_shift > exp_duration:
        if pH.nunique() > 1:
            bad_exps.append(exp)

assert not bad_exps, f"W:pH behavior did not match Z:phShift for: {bad_exps}"
print("W:pH behavior verified against Z:phShift for all experiments.")

W:pH behavior verified against Z:phShift for all experiments.


#### Validate Temperature Shift

In [ ]:
# For each experiment, verify W:temp behavior matches the scheduled Z:tempShift:
#   - 0 < tempShift <= ExpDuration: W:temp must change on that day vs. the previous day.
#   - tempShift > ExpDuration:      W:temp must stay constant throughout the experiment.
# (tempShift == 0 is excluded: no preceding day to compare to.)

bad_exps = []
for exp, exp_df in train_df.groupby("Exp"):
    setpoints = exp_df.loc[exp_df["Time[day]"] == 0].iloc[0]
    temp_shift = int(setpoints["Z:tempShift"])
    exp_duration = setpoints["Z:ExpDuration"]
    temp = exp_df.set_index("Time[day]")["W:temp"]

    if 0 < temp_shift <= exp_duration:
        if temp[temp_shift] == temp[temp_shift - 1]:
            bad_exps.append(exp)
    elif temp_shift > exp_duration:
        if temp.nunique() > 1:
            bad_exps.append(exp)

assert not bad_exps, f"W:temp behavior did not match Z:tempShift for: {bad_exps}"
print("W:temp behavior verified against Z:tempShift for all experiments.")

## Target (`Y:Titer`)

We expectd the target (final mAb concentration) to be reported **once per experiment, on its final day** (i.e. on day `Z:ExpDuration`).  
The cells below merge the targets onto the time-series and verify both invariants explicitly.

In [33]:
df = pd.merge(train_df, targets_df, how="outer", on=["Exp", "Time[day]"])

In [34]:
# Join each titer measurement with its experiment's planned duration,
# so we can compare the day on which titer was reported against Z:ExpDuration.
duration_per_exp = train_df.query("`Time[day]` == 0")[["Exp", "Z:ExpDuration"]]
titer_rows = (
    df.dropna(subset="Y:Titer")[["Exp", "Time[day]", "Y:Titer"]]
    .merge(duration_per_exp, on="Exp")
)
titer_rows

,Exp,Time[day],Y:Titer,Z:ExpDuration
0,Exp 1,10,2550.097032,10.0
1,Exp 10,14,609.903525,14.0
2,Exp 100,7,948.598303,7.0
3,Exp 11,10,1758.171425,10.0
4,Exp 12,14,3953.188470,14.0
...,...,...,...,...
95,Exp 95,8,928.520531,8.0
96,Exp 96,8,1015.602095,8.0
97,Exp 97,7,692.468630,7.0
98,Exp 98,7,310.779733,7.0


In [35]:
# Verify both target invariants:
#   1. Exactly one titer measurement per experiment.
#   2. That measurement is taken on the experiment's last day (Z:ExpDuration).
assert len(titer_rows) == num_experiments, (
    f"Expected one titer row per experiment ({num_experiments}); got {len(titer_rows)}"
)
assert (titer_rows["Time[day]"] == titer_rows["Z:ExpDuration"]).all(), (
    "Some titer measurements are not on the last day of the experiment"
)
print(f"Y:Titer recorded only on the last day, for all {num_experiments} experiments.")

Y:Titer recorded only on the last day, for all 100 experiments.


## Missing data

We've already accounted for the two groups of columns where `NaN` is structurally expected: the `Z:` setpoints (recorded only on day 0) and `Y:Titer` (recorded only on the final day).  
Every other column should be populated for every `(experiment, day)` pair. The cell below verifies this.

In [36]:
# Exclude columns where NaN is structurally expected; the rest must be fully populated.
expected_sparse_cols = setpoints_cols + ["Y:Titer"]
dense_cols = df.columns.difference(expected_sparse_cols)
nans_per_col = df[dense_cols].isna().sum()

assert (nans_per_col == 0).all(), (
    f"Unexpected NaN in: {nans_per_col[nans_per_col > 0].to_dict()}"
)
print("All non-setpoint, non-target columns are fully populated.")

All non-setpoint, non-target columns are fully populated.
